### Train a LSTM for sequence prediction


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from split.create_split import create_split
SEED = 42

2026-02-03 09:34:57.634173: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [13]:
# Load dataset: TFBIND 8 - SIX6 REF R1
df = pd.read_csv('tf_bind_8-SIX6_REF_R1/dataset.csv')
df['split'] = ['None'] * len(df)

df = create_split(df, k_seeds=50, n_training_seeds=20, random_state=42)

train_df = df[df['split'] == 'train']

x = train_df.iloc[:, :-5].values
print('Shape of x:', x.shape)
print('First row of x:', x[0])
y = train_df['binding_scores'].values

# One-hot encode and reshape to 3D for LSTM (samples, timesteps, features)
x_one_hot = np.array([tf.one_hot(seq, depth=4).numpy() for seq in x])
print('One-hot encoded shape (3D):', x_one_hot.shape)

X_train, X_val, y_train, y_val = train_test_split(x_one_hot, y, test_size=0.1, random_state=SEED)

print('Training samples: ', X_train.shape[0])
print('Validation samples: ', X_val.shape[0])
print('Input shape (timesteps, features):', X_train.shape[1:])

Shape of x: (8670, 8)
First row of x: [0 0 0 0 0 0 2 1]
One-hot encoded shape (3D): (8670, 8, 4)
Training samples:  7803
Validation samples:  867
Input shape (timesteps, features): (8, 4)


In [14]:
model = Sequential()
model.add(LSTM(100, input_shape = (X_train.shape[1:])))
model.add(Dense(4, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


/home/meier/Desktop/PracticalWork/.venv/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_5 (LSTM)                   │ (None, 100)            │        42,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           404 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,409 (165.66 KB)

 Trainable params: 42,409 (165.66 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(X_train, y_train, epochs=20, batch_size = 32, validation_data=(X_val, y_val), verbose=1)

Epoch 1/20
244/244 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0235 - mae: 0.1113 - val_loss: 0.0060 - val_mae: 0.0653
Epoch 2/20
244/244 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0102 - mae: 0.0782 - val_loss: 0.0054 - val_mae: 0.0617
Epoch 3/20
244/244 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0065 - mae: 0.0652 - val_loss: 0.0053 - val_mae: 0.0609
Epoch 4/20
244/244 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0056 - mae: 0.0609 - val_loss: 0.0053 - val_mae: 0.0596
Epoch 5/20
244/244 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0055 - mae: 0.0603 - val_loss: 0.0051 - val_mae: 0.0593
Epoch 6/20
244/244 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0054 - mae: 0.0599 - val_loss: 0.0049 - val_mae: 0.0576
Epoch 7/20
244/244 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0053 - mae: 0.0594 - val_loss: 0.0049 - val_mae: 0.0581
Epoch 8/20
244/244 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0053 - mae: 0.0592 - val_loss: 0.0049 - val_mae: 0.0584
Epoch 9/20
244/244 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - lo

In [16]:
# save trained model
model.save('lstm_sequence_model.h5')

In [17]:
# evaluate the model
test_loss = model.evaluate(X_val, y_val)
print('Test Loss:', test_loss)

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0048 - mae: 0.0571
Test Loss: [0.0047780899330973625, 0.05705997720360756]
